In [4]:
"""
Robot -> InfluxDB Logger — Neurapy
"""
import time
import os
import subprocess
import ctypes

from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import WriteOptions

ctypes.windll.winmm.timeBeginPeriod(1)
# -- Avvio container ---------------------------------------------------------
subprocess.run(["docker", "start", "mio_influxdb"], capture_output=True)
time.sleep(5)

# -- CONFIG ------------------------------------------------------------------
INFLUX_URL    = "http://localhost:8086"
INFLUX_TOKEN  = os.environ.get("INFLUXDB_TOKEN")
if not INFLUX_TOKEN:
    raise ValueError("INFLUXDB_TOKEN non trovato nelle variabili d'ambiente")
INFLUX_ORG    = "Polimi"
INFLUX_BUCKET = "KAWASAKI"

POLL_INTERVAL       = 1/100
VELOCITY_THRESHOLD  = 0.01
COUNTER_MEASUREMENT = "_trajectory_counter"

# MODIFICA 1: print di stato ogni LOG_EVERY cicli invece che ogni ciclo.
# A 50 Hz -> una riga al secondo. Elimina il costo dei print nel loop tight.
LOG_EVERY = 100

# -- Robot -------------------------------------------------------------------
try:
    from neurapy.robot import Robot
    robot = Robot()
    print("[INFO] Connesso al robot reale.")
except BaseException:
    print("[INFO] neurapy non trovato — modalità DEMO attiva.")

METRICS = [
    ("joint_angles",       robot.get_current_joint_angles_with_timestamp),
    ("joint_torques",      robot.get_current_joint_torques_with_timestamp),
    ("load_side_encoder",  robot.get_current_load_side_encoder_values_with_timestamp),
    ("motor_side_encoder", robot.get_current_motor_side_encoder_values_with_timestamp),
]

# -- InfluxDB client ---------------------------------------------------------
client    = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=WriteOptions(
    batch_size=50,
    flush_interval=500,
    jitter_interval=0
))
query_api = client.query_api()


# -- Persistenza ID traiettoria ----------------------------------------------

def get_last_trajectory_id() -> int:
    flux = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -100y)
  |> filter(fn: (r) => r._measurement == "{COUNTER_MEASUREMENT}")
  |> filter(fn: (r) => r._field == "last_id")
  |> last()
'''
    try:
        tables = query_api.query(flux, org=INFLUX_ORG)
        for table in tables:
            for record in table.records:
                return int(record.get_value())
    except Exception as e:
        print(f"[WARN] Impossibile leggere l'ultimo ID da InfluxDB: {e}")
    return 0


def save_trajectory_id(traj_id: int) -> None:
    p = Point(COUNTER_MEASUREMENT).field("last_id", traj_id)
    try:
        write_api.write(bucket=INFLUX_BUCKET, record=p)
    except Exception as e:
        print(f"[WARN] Impossibile salvare l'ID traiettoria su InfluxDB: {e}")


# -- Inizializzazione --------------------------------------------------------

last_id     = get_last_trajectory_id()
traj_id     = last_id
was_moving  = False
traj_active = False
cycle       = 0
missed      = 0          # MODIFICA 2: conta cicli che sforano il budget
t_last_log  = time.monotonic()

print(f"[INFO] Ultimo trajectory_id trovato su InfluxDB: {last_id}")
print(f"[INFO] Prossima traiettoria partirà con ID: {last_id + 1}")
print(f"[INFO] Scrittura avviata (Ctrl+C per fermare)\n")

# -- Loop principale ---------------------------------------------------------
try:
    while True:
        t0 = time.monotonic()
        points = []
        cycle += 1

        velocities, vel_ts = robot.get_current_joint_velocities_with_timestamp()
        moving = any(abs(v) > VELOCITY_THRESHOLD for v in velocities)

        # Inizio / fine traiettoria: print sempre, sono eventi rari
        if moving and not was_moving:
            traj_id     = last_id + 1
            last_id     = traj_id
            traj_active = True
            save_trajectory_id(traj_id)
            print(f"[{time.strftime('%H:%M:%S')}] >> Nuova traiettoria: ID = {traj_id}")

        elif not moving and was_moving:
            print(f"[{time.strftime('%H:%M:%S')}] == Fine traiettoria ID = {traj_id}")
            traj_active = False

        was_moving = moving

        # Scrittura dati
        if moving:
            p_vel = (Point("joint_velocities")
                     .tag("trajectory_id", str(traj_id))
                     .time(int(vel_ts), "us"))
            for i, v in enumerate(velocities):
                p_vel = p_vel.field(f"j{i+1}", float(v))
            points.append(p_vel)

            for name, fn in METRICS:
                values, ts = fn()
                p = (Point(name)
                     .tag("trajectory_id", str(traj_id))
                     .time(int(ts), "us"))
                for i, v in enumerate(values):
                    p = p.field(f"j{i+1}", float(v))
                points.append(p)

            write_api.write(bucket=INFLUX_BUCKET, record=points)

        # MODIFICA 2: rileva cicli che hanno sforato il budget
        elapsed_cycle = time.monotonic() - t0
        if elapsed_cycle > POLL_INTERVAL:
            missed += 1

        # MODIFICA 1: log periodico — Hz reali + cicli sforati nell'ultimo secondo
        if cycle % LOG_EVERY == 0:
            actual_hz = LOG_EVERY / (time.monotonic() - t_last_log)
            print(f"[{time.strftime('%H:%M:%S')}] "
                  f"Hz: {actual_hz:.1f} | "
                  f"traj: {traj_id} | "
                  f"stato: {'moving' if moving else 'fermo'} | "
                  f"cicli sforati: {missed}")
            missed     = 0
            t_last_log = time.monotonic()

        time.sleep(max(0, POLL_INTERVAL - elapsed_cycle))

except KeyboardInterrupt:
    print("\n[INFO] Stop.")
finally:
    client.close()
    ctypes.windll.winmm.timeEndPeriod(1)

[2026-03-25 14:34:15][neurapy_logger][WARNING] : Current client version is not compatiable with the version of the server running on the robot. Some of the functionlities specified in the documentation might not work in the intended way. Please upgrade to the correct version .Client Version : v4.16.4,Server Version : aaed267_v4.14.3 :(robot.py:130)
[INFO] Connesso al robot reale.
[INFO] Ultimo trajectory_id trovato su InfluxDB: 334
[INFO] Prossima traiettoria partirà con ID: 335
[INFO] Scrittura avviata (Ctrl+C per fermare)

[14:34:16] Hz: 101.6 | traj: 334 | stato: fermo | cicli sforati: 30
[14:34:17] Hz: 98.4 | traj: 334 | stato: fermo | cicli sforati: 26
[14:34:18] Hz: 94.2 | traj: 334 | stato: fermo | cicli sforati: 19
[14:34:19] Hz: 100.0 | traj: 334 | stato: fermo | cicli sforati: 28
[14:34:20] Hz: 104.9 | traj: 334 | stato: fermo | cicli sforati: 32
[14:34:21] Hz: 100.0 | traj: 334 | stato: fermo | cicli sforati: 28
[14:34:22] Hz: 85.3 | traj: 334 | stato: fermo | cicli sforati:

In [5]:
"""
Robot -> InfluxDB Logger — Neurapy
"""
import time
import os
import subprocess

print( os.environ.get("INFLUXDB_TOKEN"))

r0mw-ijYgN6zxKME8b6HrUNLfHEng9QZIZUtH0DpyzYTja8nYY5yShQVYQsCONWt5WG7c85kxWRR3AAc3er4Bg==
